In [9]:
# Cell 1 — imports
import sys
sys.path.append(".")

import pandas as pd
from modelling_utils import (
    SpatialSplitter, ClassBalancer, RFTrainer, ModelPersister, ThresholdTuner,
    FEATURE_COLUMNS, BINARY_MODEL_PATH,
)

TRAINING_CSV_PATH = "../data/output/nairobi_training_pixels_clean.csv"

In [10]:
# Cell 2 — load the cleaned dataset
df = pd.read_csv(TRAINING_CSV_PATH)
print(df.shape)
print(df["built_up"].value_counts(normalize=True))

(3898567, 12)
built_up
0    0.762241
1    0.237759
Name: proportion, dtype: float64


In [11]:
# Cell 3 — spatial train/test split (by tile, not by row)
splitter = SpatialSplitter(test_size=0.2, random_state=42)
train_df, test_df = splitter.split(df, group_col="tile_id")

print(f"Train: {len(train_df):,} rows across {train_df['tile_id'].nunique()} tiles")
print(f"Test:  {len(test_df):,} rows across {test_df['tile_id'].nunique()} tiles")

# Confirm no tile appears in both splits
overlap = set(train_df["tile_id"]) & set(test_df["tile_id"])
print(f"Tiles in both splits (should be empty): {overlap}")

Train: 2,999,242 rows across 12 tiles
Test:  899,325 rows across 4 tiles
Tiles in both splits (should be empty): set()


In [20]:
# Cell 4 — optional: subsample the training set's majority class
# (class_weight='balanced' in RFTrainer already helps too — you can use
# either approach alone, or both together, this is just giving you the option)
balancer = ClassBalancer(majority_ratio=4.0, random_state=42)
train_df_balanced = balancer.subsample(train_df, label_col="built_up")

print(f"Balanced training set: {len(train_df_balanced):,} rows")
print(train_df_balanced["built_up"].value_counts(normalize=True))

Balanced training set: 2,999,242 rows
built_up
0    0.739568
1    0.260432
Name: proportion, dtype: float64


In [21]:
# Cell 5 — train the binary RF
trainer = RFTrainer(
    feature_columns=FEATURE_COLUMNS,
    n_estimators=200,
    max_depth=20, 
    min_samples_leaf=5, 
    class_weight="balanced",
    random_state=42,
)
trainer.fit(train_df_balanced, label_col="built_up")

In [22]:
# Cell 6 — evaluate on the held-out test tiles
results = trainer.evaluate(test_df, label_col="built_up")

              precision    recall  f1-score   support

           0       0.94      0.82      0.87    753505
           1       0.43      0.71      0.54    145820

    accuracy                           0.80    899325
   macro avg       0.68      0.76      0.71    899325
weighted avg       0.85      0.80      0.82    899325

Confusion matrix:
 [[616808 136697]
 [ 42152 103668]]
Accuracy: 0.8011
F1 (weighted): 0.8188


In [23]:
# Notebook cell
from sklearn.metrics import classification_report

tuner = ThresholdTuner()
best_threshold = tuner.find_best_f1_threshold(trainer.model, test_df[trainer.feature_columns], test_df["built_up"])

y_pred_tuned = tuner.predict_at_threshold(trainer.model, test_df[trainer.feature_columns], best_threshold)
print(classification_report(test_df["built_up"], y_pred_tuned))

Best threshold: 0.595
  precision: 0.487, recall: 0.636, f1: 0.552
              precision    recall  f1-score   support

           0       0.93      0.87      0.90    753505
           1       0.49      0.64      0.55    145820

    accuracy                           0.83    899325
   macro avg       0.71      0.75      0.72    899325
weighted avg       0.85      0.83      0.84    899325



In [24]:
# Cell 7 — feature importance (which columns the RF actually relied on)
importance_df = trainer.feature_importance()
importance_df

,feature,importance
0,ndvi,0.243291
1,ndwi,0.209677
2,B4,0.180492
3,B2,0.163986
4,B3,0.080109
5,B8,0.064110
6,distance_to_river_m,0.058335


In [25]:
for depth in [10, 15, 25, 30]:
    trainer_sweep = RFTrainer(
        feature_columns=FEATURE_COLUMNS,
        n_estimators=200,
        max_depth=depth,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
    )
    trainer_sweep.fit(train_df_balanced, label_col="built_up")

    y_pred = trainer_sweep.model.predict(test_df[FEATURE_COLUMNS])
    from sklearn.metrics import precision_score, recall_score, f1_score
    p = precision_score(test_df["built_up"], y_pred)
    r = recall_score(test_df["built_up"], y_pred)
    f1 = f1_score(test_df["built_up"], y_pred)
    print(f"max_depth={depth}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")

max_depth=10: precision=0.441, recall=0.681, f1=0.535
max_depth=15: precision=0.437, recall=0.704, f1=0.539
max_depth=25: precision=0.434, recall=0.703, f1=0.537
max_depth=30: precision=0.436, recall=0.695, f1=0.536


In [17]:
# Cell 8 — pickle the trained model
# Cell 8 — updated to match the current 3-arg save()
ModelPersister.save(trainer.model, trainer.feature_columns, str(BINARY_MODEL_PATH))

Model saved to models\rf_built_up_binary.pkl (7 features, 433.3 MB)


In [18]:
# Cell 9 — sanity check: reload and confirm it predicts the same way
reloaded_model, reloaded_features = ModelPersister.load(str(BINARY_MODEL_PATH))
sample = test_df[reloaded_features].iloc[:5]
print(reloaded_model.predict(sample))

[1 1 0 1 1]


In [19]:
import os
size_mb = os.path.getsize("models/rf_built_up_binary.pkl") / (1024 * 1024)
print(f"Model file size: {size_mb:.1f} MB")

Model file size: 433.3 MB
